# Motorgehäuse in den API-Server veröffentlichen

Kernel: **SysML** (oben rechts prüfen, nicht Python).

Zelle 1 definiert das Modell, Zelle 2 schiebt es über `%publish` in den Server.
Das Ziel steuert `ISYSML_API_BASE_PATH` und zeigt auf `http://sysml-api:9000`.

In [1]:
package Test { part def Housing; }

Package Test (0a652fc5-effd-48de-abbb-44e03389e89b)


In [2]:
package Test2 { import ISQ::*; }

ERROR:mismatched input 'import' expecting '}' (2.sysml line : 1 column : 17)
ERROR:no viable alternative at input '::' (2.sysml line : 1 column : 27)


In [3]:
package Test3 { private import ISQ::*; }

Package Test3 (dce620af-04d7-4c80-ad71-eafbf6bd93cf)


In [4]:
%help

The following SysML v2 magic commands are available.
For help on a specific command, use "%help <COMMAND>" or "%<cmd> -h".

%eval		Evaluate a given expression.
%export		Save a file of the JSON representation of the abstract syntax tree rooted in the named element.
%help		Get a list of available commands or help on a specific command
%list		List loaded library packages or the results of a given query
%load		Load models from the repository
%projects	List projects in the repository
%publish	Publish to the repository the model elements rooted in a named element
%repo		Set the API base path for the repository
%show		Print the abstract syntax tree rooted in a named element
%view		Render the view specified by the named view usage
%viz		Visualize the named model elements


In [5]:
package Test {
    part def Housing;
}

Package Test (e032e3da-b168-49bf-b0df-7ad753cfe98c)


In [6]:
package MotorHousing {
    private import ScalarValues::*;
    private import ISQ::*;
    private import SI::*;

    part def Housing {
        attribute length : LengthValue;
        attribute mass   : MassValue;
    }

    requirement def MaxLength {
        doc /* Die maximale Laenge des Motorgehaeuses darf 120 mm nicht ueberschreiten. */
        subject housing : Housing;
        attribute maxLength : LengthValue = 120 [mm];
        require constraint { housing.length <= maxLength }
    }

    requirement def MaxMass {
        doc /* Die Masse des Motorgehaeuses darf 0,9 kg nicht ueberschreiten. */
        subject housing : Housing;
        attribute maxMass : MassValue = 0.9 [kg];
        require constraint { housing.mass <= maxMass }
    }

    part motorHousing : Housing;

    requirement maxLengthCheck : MaxLength;
    requirement maxMassCheck   : MaxMass;

    satisfy maxLengthCheck by motorHousing;
    satisfy maxMassCheck   by motorHousing;
}

Package MotorHousing (1b6c0aa0-6b5e-4ddf-a896-11a4832e2047)


In [7]:
%publish MotorHousing

API base path: http://sysml-api:9000
Processing.
Posting commit. Additions: 164, Deletions: 0, Modified elements: 0
Successfully posted commit: cdfddc6b-312f-475a-8fd4-324d37388cad on branch: (main) 8fd8b4e7-57be-4c03-a1ce-2a892474f7c8



Saved to Project MotorHousing (54038e0c-9056-45b6-b7ca-d7d6b1f9786a)


Wenn `%publish` eine Projekt-ID zurückgibt, liegt das Modell im Server.
Danach im Devcontainer die Elemente abrufen und die echte Payload ansehen —
das ist die Grundlage für `extract._to_requirement`:

```bash
python -c "
from sysmlcad.sysml_client import SysMLClient
c = SysMLClient(); p = c.projects(); print(p)
pid = p[0]['@id']; cid = c.head_commit(pid)
els = c.elements(pid, cid)
print(len(els), 'Elemente')
print([e['@type'] for e in els][:40])
c.close()"
```